In [5]:
# =========================================================
# Exploración de integración Spotify ↔ Grammy
# Evidencia preliminar para validar factibilidad
# =========================================================

from pathlib import Path
import re
import unicodedata

import pandas as pd
from sqlalchemy import create_engine


# ---------------------------------------------------------
# Rutas y conexión
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

SPOTIFY_PATH = PROJECT_ROOT / "data" / "spotify_dataset.csv"

DB_URL = (
    "postgresql+psycopg2://"
    "etl_user:etl_password@localhost:5433/etl_workshop"
)


# ---------------------------------------------------------
# Carga de datos
# ---------------------------------------------------------

spotify = pd.read_csv(SPOTIFY_PATH)

engine = create_engine(DB_URL)

grammy = pd.read_sql(
    "SELECT * FROM raw.grammy",
    engine
)

print("Spotify:", spotify.shape)
print("Grammy:", grammy.shape)


# ---------------------------------------------------------
# Normalización únicamente para comparación
# ---------------------------------------------------------

def normalize_text(value):
    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    value = unicodedata.normalize("NFKD", value)
    value = "".join(
        char for char in value
        if not unicodedata.combining(char)
    )

    value = re.sub(r"\s+", " ", value)

    return value


spotify["artist_match"] = spotify["artists"].apply(normalize_text)
spotify["track_match"] = spotify["track_name"].apply(normalize_text)

grammy["artist_match"] = grammy["artist"].apply(normalize_text)
grammy["nominee_match"] = grammy["nominee"].apply(normalize_text)


# ---------------------------------------------------------
# 1. Cobertura por artista
# ---------------------------------------------------------

spotify_artists = set(
    spotify["artist_match"].dropna().unique()
)

grammy_with_artist = grammy[
    grammy["artist_match"].notna()
].copy()

grammy_with_artist["artist_found"] = (
    grammy_with_artist["artist_match"]
    .isin(spotify_artists)
)

artist_matches = grammy_with_artist["artist_found"].sum()
artist_total = len(grammy_with_artist)

artist_coverage = artist_matches / artist_total * 100


# ---------------------------------------------------------
# 2. Coincidencia artista + pista
# ---------------------------------------------------------

spotify_pairs = set(
    zip(
        spotify["artist_match"],
        spotify["track_match"]
    )
)

grammy_with_track = grammy[
    grammy["artist_match"].notna()
    & grammy["nominee_match"].notna()
].copy()

grammy_with_track["track_found"] = [
    (artist, nominee) in spotify_pairs
    for artist, nominee
    in zip(
        grammy_with_track["artist_match"],
        grammy_with_track["nominee_match"]
    )
]

track_matches = grammy_with_track["track_found"].sum()
track_total = len(grammy_with_track)

track_coverage = track_matches / track_total * 100


# ---------------------------------------------------------
# 3. Ambigüedad de las coincidencias
# ---------------------------------------------------------

# Número de pistas Spotify asociadas a cada combinación
# normalizada artista + pista.

spotify_match_counts = (
    spotify
    .dropna(subset=["artist_match", "track_match"])
    .groupby(["artist_match", "track_match"])
    .size()
    .reset_index(name="spotify_records")
)

matched_pairs = grammy_with_track[
    grammy_with_track["track_found"]
].merge(
    spotify_match_counts,
    left_on=["artist_match", "nominee_match"],
    right_on=["artist_match", "track_match"],
    how="left"
)

one_to_one = (
    matched_pairs["spotify_records"] == 1
).sum()

one_to_many = (
    matched_pairs["spotify_records"] > 1
).sum()

no_match = (
    ~grammy_with_track["track_found"]
).sum()


# ---------------------------------------------------------
# Resultados
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("EVIDENCIA DE INTEGRACIÓN SPOTIFY ↔ GRAMMY")
print("=" * 60)

print("\n--- COBERTURA POR ARTISTA ---")
print(f"Registros Grammy con artista: {artist_total}")
print(f"Con coincidencia en Spotify: {artist_matches}")
print(f"Cobertura: {artist_coverage:.2f}%")

print("\n--- COBERTURA POR ARTISTA + PISTA ---")
print(f"Registros Grammy con artista + nominee: {track_total}")
print(f"Con coincidencia en Spotify: {track_matches}")
print(f"Cobertura: {track_coverage:.2f}%")

print("\n--- AMBIGÜEDAD DE LAS COINCIDENCIAS ---")
print(f"Coincidencias 1:1: {one_to_one}")
print(f"Coincidencias 1:N: {one_to_many}")
print(f"Sin coincidencia: {no_match}")

print("\n--- VARIABLES DISPONIBLES PARA ANÁLISIS ---")
print("Spotify:")
print("  - popularity")
print("  - track_genre")
print("  - danceability")
print("  - energy")
print("  - valence")
print("  - acousticness")
print("  - instrumentalness")
print("  - tempo")

print("\nGrammy:")
print("  - year")
print("  - category")
print("  - winner")
print("  - artist")
print("  - nominee")

Spotify: (114000, 21)
Grammy: (4810, 10)

EVIDENCIA DE INTEGRACIÓN SPOTIFY ↔ GRAMMY

--- COBERTURA POR ARTISTA ---
Registros Grammy con artista: 2970
Con coincidencia en Spotify: 1018
Cobertura: 34.28%

--- COBERTURA POR ARTISTA + PISTA ---
Registros Grammy con artista + nominee: 2970
Con coincidencia en Spotify: 181
Cobertura: 6.09%

--- AMBIGÜEDAD DE LAS COINCIDENCIAS ---
Coincidencias 1:1: 86
Coincidencias 1:N: 95
Sin coincidencia: 2789

--- VARIABLES DISPONIBLES PARA ANÁLISIS ---
Spotify:
  - popularity
  - track_genre
  - danceability
  - energy
  - valence
  - acousticness
  - instrumentalness
  - tempo

Grammy:
  - year
  - category
  - winner
  - artist
  - nominee


In [6]:
# =========================================================
# Prueba 2 — Matching Spotify ↔ Grammy
# Evaluación de artista + pista + álbum
# =========================================================

from pathlib import Path
import re
import unicodedata

import pandas as pd
from sqlalchemy import create_engine


# ---------------------------------------------------------
# Carga
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

SPOTIFY_PATH = PROJECT_ROOT / "data" / "spotify_dataset.csv"

DB_URL = (
    "postgresql+psycopg2://"
    "etl_user:etl_password@localhost:5433/etl_workshop"
)

spotify = pd.read_csv(SPOTIFY_PATH)

engine = create_engine(DB_URL)

grammy = pd.read_sql(
    "SELECT * FROM raw.grammy",
    engine
)


# ---------------------------------------------------------
# Normalización para matching
# ---------------------------------------------------------

def normalize_text(value):
    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    value = unicodedata.normalize("NFKD", value)
    value = "".join(
        char for char in value
        if not unicodedata.combining(char)
    )

    value = re.sub(r"\s+", " ", value)

    return value


spotify["artist_norm"] = spotify["artists"].apply(normalize_text)
spotify["track_norm"] = spotify["track_name"].apply(normalize_text)
spotify["album_norm"] = spotify["album_name"].apply(normalize_text)

grammy["artist_norm"] = grammy["artist"].apply(normalize_text)
grammy["track_norm"] = grammy["nominee"].apply(normalize_text)


# ---------------------------------------------------------
# 1. Matching de artista:
#    exacto vs artista contenido en campo multivaluado
# ---------------------------------------------------------

spotify_artists = spotify["artist_norm"].dropna().unique()

grammy_artists = grammy[
    grammy["artist_norm"].notna()
].copy()

grammy_artists["artist_exact"] = (
    grammy_artists["artist_norm"].isin(spotify_artists)
)


def artist_in_spotify(grammy_artist):
    if pd.isna(grammy_artist):
        return False

    for spotify_artist in spotify_artists:
        if pd.isna(spotify_artist):
            continue

        artists = [
            artist.strip()
            for artist in spotify_artist.split(";")
        ]

        if grammy_artist in artists:
            return True

    return False


grammy_artists["artist_multivalue"] = (
    grammy_artists["artist_norm"].apply(artist_in_spotify)
)


# ---------------------------------------------------------
# 2. Matching artista + pista
# ---------------------------------------------------------

spotify_artist_track = set(
    zip(
        spotify["artist_norm"],
        spotify["track_norm"]
    )
)

grammy_track = grammy[
    grammy["artist_norm"].notna()
    & grammy["track_norm"].notna()
].copy()

grammy_track["artist_track_exact"] = [
    (artist, track) in spotify_artist_track
    for artist, track
    in zip(
        grammy_track["artist_norm"],
        grammy_track["track_norm"]
    )
]


# ---------------------------------------------------------
# 3. Matching utilizando artista multivaluado
# ---------------------------------------------------------

spotify_artist_track_rows = []

for _, row in spotify[
    ["artist_norm", "track_norm"]
].dropna().iterrows():

    artists = [
        artist.strip()
        for artist in row["artist_norm"].split(";")
    ]

    for artist in artists:
        spotify_artist_track_rows.append(
            (artist, row["track_norm"])
        )

spotify_artist_track_multivalue = set(
    spotify_artist_track_rows
)

grammy_track["artist_track_multivalue"] = [
    (artist, track) in spotify_artist_track_multivalue
    for artist, track
    in zip(
        grammy_track["artist_norm"],
        grammy_track["track_norm"]
    )
]


# ---------------------------------------------------------
# 4. Analizar coincidencias y aporte del álbum
#
# Grammy no tiene un campo album_name.
# Por tanto, el álbum NO puede utilizarse para hacer
# matching directo Grammy ↔ Spotify.
#
# Lo evaluamos únicamente como elemento de
# desambiguación dentro de las coincidencias Spotify.
# ---------------------------------------------------------

# Para cada artista + pista en Spotify calculamos
# cuántos álbumes diferentes existen.

track_album_counts = (
    spotify[
        ["artist_norm", "track_norm", "album_norm"]
    ]
    .dropna(subset=["artist_norm", "track_norm"])
    .drop_duplicates()
    .groupby(["artist_norm", "track_norm"])
    .agg(
        spotify_records=("album_norm", "size"),
        different_albums=("album_norm", "nunique")
    )
    .reset_index()
)


# ---------------------------------------------------------
# 5. Resultados generales
# ---------------------------------------------------------

total_artists = len(grammy_artists)

exact_artist = grammy_artists["artist_exact"].sum()
multivalue_artist = grammy_artists["artist_multivalue"].sum()

total_tracks = len(grammy_track)

exact_artist_track = (
    grammy_track["artist_track_exact"].sum()
)

multivalue_artist_track = (
    grammy_track["artist_track_multivalue"].sum()
)

additional_artist_matches = (
    multivalue_artist - exact_artist
)

additional_track_matches = (
    multivalue_artist_track - exact_artist_track
)


print("=" * 65)
print("PRUEBA 2 — MATCHING SPOTIFY ↔ GRAMMY")
print("=" * 65)

print("\n--- ARTISTA ---")
print(f"Registros Grammy con artista: {total_artists}")

print(
    f"Coincidencia exacta: "
    f"{exact_artist} "
    f"({exact_artist / total_artists * 100:.2f}%)"
)

print(
    f"Coincidencia dentro de artistas multivaluados: "
    f"{multivalue_artist} "
    f"({multivalue_artist / total_artists * 100:.2f}%)"
)

print(
    f"Coincidencias adicionales: "
    f"{additional_artist_matches}"
)


print("\n--- ARTISTA + PISTA ---")
print(f"Registros Grammy con artista + nominee: {total_tracks}")

print(
    f"Coincidencia exacta: "
    f"{exact_artist_track} "
    f"({exact_artist_track / total_tracks * 100:.2f}%)"
)

print(
    f"Coincidencia con artista multivaluado: "
    f"{multivalue_artist_track} "
    f"({multivalue_artist_track / total_tracks * 100:.2f}%)"
)

print(
    f"Coincidencias adicionales: "
    f"{additional_track_matches}"
)


# ---------------------------------------------------------
# 6. Aporte potencial del álbum
# ---------------------------------------------------------

album_summary = track_album_counts[
    track_album_counts["different_albums"] > 1
]

print("\n--- APORTE DEL ÁLBUM ---")

print(
    "Combinaciones artista + pista con múltiples álbumes: "
    f"{len(album_summary)}"
)

print(
    "Combinaciones artista + pista con un único álbum: "
    f"{len(track_album_counts) - len(album_summary)}"
)

if len(track_album_counts) > 0:
    ambiguous_album_pct = (
        len(album_summary)
        / len(track_album_counts)
        * 100
    )

    print(
        f"Porcentaje donde el álbum podría ayudar a "
        f"desambiguar: {ambiguous_album_pct:.2f}%"
    )


# ---------------------------------------------------------
# 7. Mostrar ejemplos donde el álbum podría ser relevante
# ---------------------------------------------------------

if len(album_summary) > 0:

    examples = (
        spotify[
            ["artist_norm", "track_norm", "album_norm",
             "artists", "track_name", "album_name"]
        ]
        .merge(
            album_summary[
                ["artist_norm", "track_norm"]
            ],
            on=["artist_norm", "track_norm"],
            how="inner"
        )
        .sort_values(
            ["artist_norm", "track_norm"]
        )
        .head(20)
    )

    print("\n--- EJEMPLOS DE POSIBLE DESAMBIGUACIÓN ---")

    display(
        examples[
            [
                "artists",
                "track_name",
                "album_name"
            ]
        ]
    )

PRUEBA 2 — MATCHING SPOTIFY ↔ GRAMMY

--- ARTISTA ---
Registros Grammy con artista: 2970
Coincidencia exacta: 1018 (34.28%)
Coincidencia dentro de artistas multivaluados: 1310 (44.11%)
Coincidencias adicionales: 292

--- ARTISTA + PISTA ---
Registros Grammy con artista + nominee: 2970
Coincidencia exacta: 181 (6.09%)
Coincidencia con artista multivaluado: 188 (6.33%)
Coincidencias adicionales: 7

--- APORTE DEL ÁLBUM ---
Combinaciones artista + pista con múltiples álbumes: 4510
Combinaciones artista + pista con un único álbum: 76666
Porcentaje donde el álbum podría ayudar a desambiguar: 5.56%

--- EJEMPLOS DE POSIBLE DESAMBIGUACIÓN ---


,artists,track_name,album_name
7767,2 Brothers On The 4th Floor,Dreams (Will Come Alive),Dreams
7769,2 Brothers On The 4th Floor,Dreams (Will Come Alive),Dreams (Will Come Alive)
7809,2 Brothers On The 4th Floor,Fairytales - Charly Lownoise & Mental Theo Rav...,The Very Best of 2 Brothers on the 4th Floor (...
7852,2 Brothers On The 4th Floor,Fairytales - Charly Lownoise & Mental Theo Rav...,2
7787,2 Brothers On The 4th Floor,Fly,Fly (Through The Starry Night)
7832,2 Brothers On The 4th Floor,Fly,The Very Best of 2 Brothers on the 4th Floor (...
15656,22Gz,Fallen Blixkys,New Grooves
15657,22Gz,Fallen Blixkys,Drillin' & Rappin'
15658,22Gz,Fallen Blixkys,Trending Hip Hop
15655,22Gz,"Suburban, Pt. 2",The Hype


In [11]:
# =========================================================
# Prueba 3 — Cardinalidad de integración Grammy ↔ Spotify
# Nivel: artista
# =========================================================

# ---------------------------------------------------------
# 1. Preparar artistas Grammy
# ---------------------------------------------------------

grammy_artists = (
    grammy[
        grammy["artist_norm"].notna()
    ][
        ["artist_norm", "artist"]
    ]
    .drop_duplicates()
    .copy()
)


# ---------------------------------------------------------
# 2. Separar artistas multivaluados de Spotify
# ---------------------------------------------------------

spotify_exploded = spotify[
    [
        "artist_norm",
        "track_norm",
        "track_name",
        "album_name",
        "track_genre",
        "popularity"
    ]
].dropna(
    subset=["artist_norm"]
).copy()

spotify_exploded["artist_individual"] = (
    spotify_exploded["artist_norm"]
    .str.split(";")
)

spotify_exploded = (
    spotify_exploded
    .explode("artist_individual")
)

spotify_exploded["artist_individual"] = (
    spotify_exploded["artist_individual"]
    .str.strip()
)


# ---------------------------------------------------------
# 3. Identificar artistas Grammy presentes en Spotify
# ---------------------------------------------------------

spotify_artist_set = set(
    spotify_exploded[
        "artist_individual"
    ].dropna()
)

grammy_artists["spotify_match"] = (
    grammy_artists["artist_norm"]
    .isin(spotify_artist_set)
)

matched_grammy_artists = (
    grammy_artists[
        grammy_artists["spotify_match"]
    ]
    .copy()
)


# ---------------------------------------------------------
# 4. Contar pistas Spotify por artista
#
# Se agrupa primero Spotify y después se une el resultado
# con Grammy. Así evitamos columnas duplicadas.
# ---------------------------------------------------------

spotify_artist_summary = (
    spotify_exploded
    .groupby(
        "artist_individual"
    )
    .agg(
        spotify_tracks=(
            "track_norm",
            "nunique"
        ),
        spotify_records=(
            "track_name",
            "size"
        ),
        spotify_genres=(
            "track_genre",
            "nunique"
        )
    )
    .reset_index()
)


artist_cardinality_summary = (
    matched_grammy_artists[
        ["artist_norm", "artist"]
    ]
    .merge(
        spotify_artist_summary,
        left_on="artist_norm",
        right_on="artist_individual",
        how="left"
    )
    .drop(
        columns=["artist_individual"]
    )
)


# ---------------------------------------------------------
# 5. Distribución de cardinalidad
# ---------------------------------------------------------

cardinality_distribution = (
    artist_cardinality_summary[
        "spotify_tracks"
    ]
    .describe()
)

print("=" * 65)
print("PRUEBA 3 — CARDINALIDAD GRAMMY ↔ SPOTIFY")
print("=" * 65)

print("\n--- ARTISTAS ---")

print(
    f"Artistas Grammy con artista válido: "
    f"{len(grammy_artists)}"
)

print(
    f"Artistas Grammy con coincidencia Spotify: "
    f"{len(matched_grammy_artists)}"
)

print(
    f"Cobertura: "
    f"{len(matched_grammy_artists) / len(grammy_artists) * 100:.2f}%"
)


print("\n--- PISTAS SPOTIFY POR ARTISTA ---")

print(cardinality_distribution)


# ---------------------------------------------------------
# 6. Distribución de cardinalidad
# ---------------------------------------------------------

artist_cardinality_summary["cardinality_group"] = pd.cut(
    artist_cardinality_summary[
        "spotify_tracks"
    ],
    bins=[
        0,
        1,
        5,
        10,
        25,
        50,
        100,
        float("inf")
    ],
    labels=[
        "1",
        "2-5",
        "6-10",
        "11-25",
        "26-50",
        "51-100",
        "101+"
    ],
    include_lowest=True
)

cardinality_counts = (
    artist_cardinality_summary[
        "cardinality_group"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\n--- DISTRIBUCIÓN DE ARTISTAS "
    "SEGÚN NÚMERO DE PISTAS ---"
)

print(cardinality_counts)


# ---------------------------------------------------------
# 7. Cobertura del dataset Spotify
# ---------------------------------------------------------

matched_spotify_tracks = (
    spotify_exploded[
        spotify_exploded[
            "artist_individual"
        ].isin(
            set(
                matched_grammy_artists[
                    "artist_norm"
                ]
            )
        )
    ][
        "track_norm"
    ]
    .dropna()
    .nunique()
)

total_spotify_tracks = (
    spotify["track_id"].nunique()
)

print("\n--- COBERTURA DE SPOTIFY ---")

print(
    f"Pistas Spotify asociadas a artistas Grammy: "
    f"{matched_spotify_tracks}"
)

print(
    f"Pistas Spotify totales: "
    f"{total_spotify_tracks}"
)

print(
    f"Cobertura de pistas: "
    f"{matched_spotify_tracks / total_spotify_tracks * 100:.2f}%"
)


# ---------------------------------------------------------
# 8. Artistas con mayor cardinalidad
# ---------------------------------------------------------

print(
    "\n--- 20 ARTISTAS CON MÁS PISTAS SPOTIFY ---"
)

display(
    artist_cardinality_summary
    .sort_values(
        "spotify_tracks",
        ascending=False
    )
    .head(20)
)


# ---------------------------------------------------------
# 9. Artistas con una sola pista
# ---------------------------------------------------------

print(
    "\n--- ARTISTAS CON UNA SOLA PISTA ---"
)

display(
    artist_cardinality_summary[
        artist_cardinality_summary[
            "spotify_tracks"
        ] == 1
    ]
    .head(20)
)


# ---------------------------------------------------------
# 10. Contrato de integración
# ---------------------------------------------------------

print("\n--- CONTRATO DE INTEGRACIÓN ---")

print(
    "Origen: artista registrado en Grammy"
)

print(
    "Destino: pistas asociadas al artista en Spotify"
)

print(
    "Cardinalidad: 1:N"
)

PRUEBA 3 — CARDINALIDAD GRAMMY ↔ SPOTIFY

--- ARTISTAS ---
Artistas Grammy con artista válido: 1658
Artistas Grammy con coincidencia Spotify: 534
Cobertura: 32.21%

--- PISTAS SPOTIFY POR ARTISTA ---
count    534.000000
mean       9.765918
std       18.374249
min        1.000000
25%        2.000000
50%        4.000000
75%       10.000000
max      268.000000
Name: spotify_tracks, dtype: float64

--- DISTRIBUCIÓN DE ARTISTAS SEGÚN NÚMERO DE PISTAS ---
cardinality_group
1         119
2-5       192
6-10       90
11-25      86
26-50      33
51-100     11
101+        3
Name: count, dtype: int64

--- COBERTURA DE SPOTIFY ---
Pistas Spotify asociadas a artistas Grammy: 4754
Pistas Spotify totales: 89741
Cobertura de pistas: 5.30%

--- 20 ARTISTAS CON MÁS PISTAS SPOTIFY ---


,artist_norm,artist,spotify_tracks,spotify_records,spotify_genres,cardinality_group
356,george jones,George Jones,268,343,1,101+
140,the beatles,The Beatles,150,280,3,101+
334,linkin park,Linkin Park,104,253,4,101+
59,chris tomlin,Chris Tomlin,92,127,3,51-100
409,pink floyd,Pink Floyd,84,89,2,51-100
502,elvis presley,Elvis Presley,83,175,3,51-100
90,tove lo,Tove Lo,69,133,12,51-100
12,andrea bocelli,Andrea Bocelli,69,87,3,51-100
17,the chemical brothers,The Chemical Brothers,69,90,4,51-100
67,bad bunny,Bad Bunny,63,416,7,51-100



--- ARTISTAS CON UNA SOLA PISTA ---


,artist_norm,artist,spotify_tracks,spotify_records,spotify_genres,cardinality_group
3,h.e.r.,H.E.R.,1,3,1,1
8,vampire weekend,Vampire Weekend,1,1,1,1
21,flume,Flume,1,2,2,1
22,rodrigo y gabriela,Rodrigo y Gabriela,1,1,1,1
34,bj the chicago kid,BJ The Chicago Kid,1,2,2,1
35,nao,NAO,1,1,1,1
40,"tyler, the creator","Tyler, The Creator",1,1,1,1
54,brad mehldau,Brad Mehldau,1,2,2,1
55,kirk franklin,Kirk Franklin,1,2,1,1
57,danny gokey,Danny Gokey,1,1,1,1



--- CONTRATO DE INTEGRACIÓN ---
Origen: artista registrado en Grammy
Destino: pistas asociadas al artista en Spotify
Cardinalidad: 1:N


In [13]:
# ============================================================
# PRUEBA 4 — CARDINALIDAD REAL: REGISTRO GRAMMY → SPOTIFY
# ============================================================

import pandas as pd
import re
from pathlib import Path

# ------------------------------------------------------------
# 1. Cargar datasets
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

spotify = pd.read_csv(
    PROJECT_ROOT / "data" / "spotify_dataset.csv"
)

grammy = pd.read_csv(
    PROJECT_ROOT / "data" / "the_grammy_awards.csv",
    sep=";"
)

# ID técnico del registro Grammy
grammy["grammy_record_id"] = range(1, len(grammy) + 1)


# ------------------------------------------------------------
# 2. Normalización de nombres de artista
# ------------------------------------------------------------

def normalize_text(value):
    if pd.isna(value):
        return None

    value = str(value).lower().strip()
    value = re.sub(r"\s+", " ", value)

    return value


spotify["artists_norm"] = spotify["artists"].apply(normalize_text)
grammy["artist_norm"] = grammy["artist"].apply(normalize_text)


# ------------------------------------------------------------
# 3. Separar artistas multivaluados de Spotify
# ------------------------------------------------------------

spotify_artists = spotify[
    ["track_id", "artists_norm"]
].dropna().copy()

spotify_artists["artist_individual"] = (
    spotify_artists["artists_norm"].str.split(";")
)

spotify_artists = spotify_artists.explode(
    "artist_individual"
)

spotify_artists["artist_individual"] = (
    spotify_artists["artist_individual"].str.strip()
)

spotify_artists = spotify_artists[
    ["track_id", "artist_individual"]
].drop_duplicates()


# ------------------------------------------------------------
# 4. Registros Grammy con artista válido
# ------------------------------------------------------------

grammy_artists = grammy[
    ["grammy_record_id", "artist_norm"]
].dropna().copy()

grammy_artists = grammy_artists[
    grammy_artists["artist_norm"].str.strip() != ""
]


# ------------------------------------------------------------
# 5. Join Grammy → Spotify
# ------------------------------------------------------------

matches = grammy_artists.merge(
    spotify_artists,
    left_on="artist_norm",
    right_on="artist_individual",
    how="inner"
)


# ------------------------------------------------------------
# 6. Métricas principales
# ------------------------------------------------------------

total_grammy_records = len(grammy)

grammy_records_with_artist = (
    grammy["artist"].notna().sum()
)

grammy_records_matched = (
    matches["grammy_record_id"].nunique()
)

distinct_spotify_tracks = (
    matches["track_id"].nunique()
)

join_rows = len(matches)

tracks_per_grammy = (
    matches.groupby("grammy_record_id")["track_id"]
    .nunique()
)

avg_tracks_per_grammy = tracks_per_grammy.mean()
median_tracks_per_grammy = tracks_per_grammy.median()
max_tracks_per_grammy = tracks_per_grammy.max()


# ------------------------------------------------------------
# 7. Distribución de cardinalidad
# ------------------------------------------------------------

cardinality_distribution = (
    tracks_per_grammy
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 8. Tracks asociados a múltiples registros Grammy
# ------------------------------------------------------------

grammy_records_per_track = (
    matches.groupby("track_id")["grammy_record_id"]
    .nunique()
)

tracks_multiple_grammy = (
    grammy_records_per_track > 1
).sum()

max_grammy_per_track = (
    grammy_records_per_track.max()
)


# ------------------------------------------------------------
# 9. Factor potencial de duplicación
# ------------------------------------------------------------

duplication_factor = (
    join_rows / distinct_spotify_tracks
    if distinct_spotify_tracks > 0
    else 0
)


# ------------------------------------------------------------
# 10. Resultados
# ------------------------------------------------------------

print("=" * 65)
print("PRUEBA 4 — CARDINALIDAD REAL GRAMMY → SPOTIFY")
print("=" * 65)

print("\n--- COBERTURA ---")
print(f"Registros Grammy totales:              {total_grammy_records}")
print(f"Registros Grammy con artista:          {grammy_records_with_artist}")
print(f"Registros Grammy con match Spotify:    {grammy_records_matched}")

if grammy_records_with_artist > 0:
    coverage = (
        grammy_records_matched /
        grammy_records_with_artist * 100
    )
    print(f"Cobertura de registros Grammy:         {coverage:.2f}%")

print("\n--- CARDINALIDAD GRAMMY → SPOTIFY ---")
print(f"Filas generadas por el JOIN:           {join_rows}")
print(f"Track IDs Spotify distintos:           {distinct_spotify_tracks}")
print(f"Promedio de pistas por registro:       {avg_tracks_per_grammy:.2f}")
print(f"Mediana de pistas por registro:        {median_tracks_per_grammy:.2f}")
print(f"Máximo de pistas por registro:         {max_tracks_per_grammy}")

print("\n--- DISTRIBUCIÓN ---")
print(cardinality_distribution)

print("\n--- POSIBLE DUPLICACIÓN ---")
print(
    f"Tracks Spotify vinculados a >1 Grammy: "
    f"{tracks_multiple_grammy}"
)
print(
    f"Máximo de registros Grammy por track: "
    f"{max_grammy_per_track}"
)
print(
    f"Factor filas JOIN / tracks distintos: "
    f"{duplication_factor:.2f}"
)

print("\n" + "=" * 65)

PRUEBA 4 — CARDINALIDAD REAL GRAMMY → SPOTIFY

--- COBERTURA ---
Registros Grammy totales:              4810
Registros Grammy con artista:          2970
Registros Grammy con match Spotify:    1295
Cobertura de registros Grammy:         43.60%

--- CARDINALIDAD GRAMMY → SPOTIFY ---
Filas generadas por el JOIN:           20286
Track IDs Spotify distintos:           6567
Promedio de pistas por registro:       15.66
Mediana de pistas por registro:        6.00
Máximo de pistas por registro:         332

--- DISTRIBUCIÓN ---
track_id
1      214
2      155
3      120
4       69
5       71
      ... 
129      1
136     13
150      7
154      1
332      2
Name: count, Length: 71, dtype: int64

--- POSIBLE DUPLICACIÓN ---
Tracks Spotify vinculados a >1 Grammy: 3670
Máximo de registros Grammy por track: 21
Factor filas JOIN / tracks distintos: 3.09



In [16]:
# ============================================================
# PRUEBA 5 — CAUSA DE LA MULTIPLICIDAD GRAMMY ↔ SPOTIFY
# ============================================================

# ------------------------------------------------------------
# 1. Incorporar información del registro Grammy
# ------------------------------------------------------------

match_detail = matches.merge(
    grammy[
        [
            "grammy_record_id",
            "year",
            "category",
            "nominee",
            "artist"
        ]
    ],
    on="grammy_record_id",
    how="left"
)


# ------------------------------------------------------------
# 2. Multiplicidad por track Spotify
# ------------------------------------------------------------

track_multiplicity = (
    match_detail
    .groupby("track_id")
    .agg(
        grammy_records=("grammy_record_id", "nunique"),
        years=("year", "nunique"),
        categories=("category", "nunique"),
        year_category=("grammy_record_id", "count")
    )
    .sort_values(
        "grammy_records",
        ascending=False
    )
)

print("=" * 70)
print("PRUEBA 5 — CAUSA DE LA MULTIPLICIDAD")
print("=" * 70)


# ------------------------------------------------------------
# 3. Distribución del número de registros Grammy por track
# ------------------------------------------------------------

print("\n--- DISTRIBUCIÓN DE REGISTROS GRAMMY POR TRACK ---")

print(
    track_multiplicity["grammy_records"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 4. Resumen general
# ------------------------------------------------------------

print("\n--- RESUMEN ---")

print(
    f"Tracks Spotify analizados: "
    f"{len(track_multiplicity)}"
)

print(
    f"Tracks vinculados a >1 registro Grammy: "
    f"{(track_multiplicity['grammy_records'] > 1).sum()}"
)

print(
    f"Tracks vinculados a >1 año Grammy: "
    f"{(track_multiplicity['years'] > 1).sum()}"
)

print(
    f"Tracks vinculados a >1 categoría: "
    f"{(track_multiplicity['categories'] > 1).sum()}"
)


# ------------------------------------------------------------
# 5. Top 20 tracks con mayor multiplicidad
# ------------------------------------------------------------

print("\n--- TOP 20 TRACKS CON MAYOR MULTIPLICIDAD ---")

top_tracks = (
    match_detail[
        [
            "track_id",
            "artist",
            "grammy_record_id",
            "year",
            "category",
            "nominee"
        ]
    ]
    .drop_duplicates()
    .groupby("track_id")
    .agg(
        grammy_records=("grammy_record_id", "nunique"),
        years=("year", lambda x: ", ".join(
            sorted(x.astype(str).unique())
        )),
        categories=("category", lambda x: " | ".join(
            sorted(x.dropna().astype(str).unique())
        )),
        artist=("artist", "first")
    )
    .sort_values(
        "grammy_records",
        ascending=False
    )
    .head(20)
)

print(top_tracks.to_string())


# ------------------------------------------------------------
# 6. ¿La multiplicidad se explica por años, categorías
#    o ambas?
# ------------------------------------------------------------

print("\n--- PATRONES DE MULTIPLICIDAD ---")

multiple = track_multiplicity[
    track_multiplicity["grammy_records"] > 1
]

print(
    f"Tracks con múltiples registros Grammy: "
    f"{len(multiple)}"
)

print(
    f"  Múltiples años: "
    f"{(multiple['years'] > 1).sum()}"
)

print(
    f"  Múltiples categorías: "
    f"{(multiple['categories'] > 1).sum()}"
)

print(
    f"  Múltiples años Y categorías: "
    f"{((multiple['years'] > 1) & (multiple['categories'] > 1)).sum()}"
)

print("\n" + "=" * 70)

PRUEBA 5 — CAUSA DE LA MULTIPLICIDAD

--- DISTRIBUCIÓN DE REGISTROS GRAMMY POR TRACK ---
grammy_records
1     2897
2     1223
3      625
4      471
5      262
6      300
7      215
8       92
9       72
10      42
11      87
12      17
13     211
14      18
15       5
16       9
17       1
18      15
20       2
21       3
Name: count, dtype: int64

--- RESUMEN ---
Tracks Spotify analizados: 6567
Tracks vinculados a >1 registro Grammy: 3670
Tracks vinculados a >1 año Grammy: 3306
Tracks vinculados a >1 categoría: 3100

--- TOP 20 TRACKS CON MAYOR MULTIPLICIDAD ---
                        grammy_records                                                                                           years                                                                                                                                                                                                                                                                                                        

In [17]:
# ============================================================
# PRUEBA 6 — VALIDACIÓN CUALITATIVA DE MATCHES MÚLTIPLES
# ============================================================

# ------------------------------------------------------------
# 1. Contar relaciones Grammy por track
# ------------------------------------------------------------

track_counts = (
    match_detail
    .groupby("track_id")["grammy_record_id"]
    .nunique()
    .sort_values(ascending=False)
)

# Seleccionar los tracks con mayor multiplicidad
top_track_ids = track_counts.head(10).index


# ------------------------------------------------------------
# 2. Mostrar detalle de cada track
# ------------------------------------------------------------

for track_id in top_track_ids:

    subset = (
        match_detail[
            match_detail["track_id"] == track_id
        ]
        [
            [
                "track_id",
                "artist",
                "grammy_record_id",
                "year",
                "category",
                "nominee"
            ]
        ]
        .drop_duplicates()
        .sort_values(["year", "category"])
    )

    print("\n" + "=" * 80)
    print(f"TRACK ID: {track_id}")
    print(
        f"Registros Grammy asociados: "
        f"{subset['grammy_record_id'].nunique()}"
    )
    print("=" * 80)

    print(
        subset.to_string(index=False)
    )


TRACK ID: 4kyKnNbxFe86MMeV4xpDL8
Registros Grammy asociados: 21
              track_id       artist  grammy_record_id  year                             category                                nominee
4kyKnNbxFe86MMeV4xpDL8    B.B. King              4287  1970     Best R&B Vocal Performance, Male                     The Thrill Is Gone
4kyKnNbxFe86MMeV4xpDL8    B.B. King              3735  1981 Best Ethnic Or Traditional Recording There Must Be A Better World Somewhere
4kyKnNbxFe86MMeV4xpDL8    B.B. King              3609  1983     Best Traditional Blues Recording                         Blues 'N' Jazz
4kyKnNbxFe86MMeV4xpDL8    B.B. King              3475  1985     Best Traditional Blues Recording              My Guitar Sings The Blues
4kyKnNbxFe86MMeV4xpDL8 Eric Clapton              3076  1990    Best Rock Vocal Performance, Male                               Bad Love
4kyKnNbxFe86MMeV4xpDL8    B.B. King              3111  1990     Best Traditional Blues Recording                    Liv

In [18]:
# ============================================================
# PRUEBA 7 — COBERTURA ADICIONAL MEDIANTE WORKERS
# ============================================================

import pandas as pd
import re

# ------------------------------------------------------------
# 1. Funciones de normalización
# ------------------------------------------------------------

def normalize_person(value):
    if pd.isna(value):
        return None

    value = str(value).lower().strip()
    value = re.sub(r"\s+", " ", value)

    return value


# ------------------------------------------------------------
# 2. Registros Grammy sin artist pero con workers
# ------------------------------------------------------------

workers_only = grammy[
    grammy["artist"].isna() &
    grammy["workers"].notna()
].copy()

print("=" * 70)
print("PRUEBA 7 — COBERTURA ADICIONAL MEDIANTE WORKERS")
print("=" * 70)

print("\n--- DISPONIBILIDAD DE WORKERS ---")

print(
    f"Registros Grammy sin artist: "
    f"{grammy['artist'].isna().sum()}"
)

print(
    f"Registros sin artist pero con workers: "
    f"{len(workers_only)}"
)

print(
    f"Registros sin artist ni workers: "
    f"{len(grammy[
        grammy['artist'].isna() &
        grammy['workers'].isna()
    ])}"
)


# ------------------------------------------------------------
# 3. Extraer personas de workers
#
# Formato esperado:
# "Nombre; rol; Nombre; rol"
#
# También se contemplan separadores internos "&".
# ------------------------------------------------------------

worker_records = []

for _, row in workers_only.iterrows():

    workers_text = row["workers"]

    # Separar primero los grupos definidos por ";"
    entries = [
        entry.strip()
        for entry in str(workers_text).split(";")
        if entry.strip()
    ]

    for entry in entries:

        # Intentar separar nombre y rol
        parts = entry.rsplit(",", 1)

        if len(parts) == 2:
            person = parts[0].strip()
            role = parts[1].strip()
        else:
            person = entry.strip()
            role = None

        # Si existen múltiples personas conectadas por &
        persons = re.split(r"\s*&\s*", person)

        for individual in persons:

            individual = individual.strip()

            if individual:
                worker_records.append({
                    "grammy_record_id": row["grammy_record_id"],
                    "year": row["year"],
                    "category": row["category"],
                    "nominee": row["nominee"],
                    "worker_raw": individual,
                    "role": role,
                    "worker_norm": normalize_person(individual)
                })


workers_exploded = pd.DataFrame(worker_records)


# ------------------------------------------------------------
# 4. Mostrar estructura obtenida
# ------------------------------------------------------------

print("\n--- PERSONAS EXTRAÍDAS DE WORKERS ---")

print(
    f"Registros de personas/roles extraídos: "
    f"{len(workers_exploded)}"
)

print(
    f"Personas normalizadas distintas: "
    f"{workers_exploded['worker_norm'].nunique()}"
)

print(
    "\nDistribución de roles:"
)

print(
    workers_exploded["role"]
    .value_counts(dropna=False)
    .head(20)
)


# ------------------------------------------------------------
# 5. Crear catálogo de artistas Spotify
# ------------------------------------------------------------

spotify_artist_catalog = (
    spotify_artists[
        ["artist_individual"]
    ]
    .drop_duplicates()
    .copy()
)

spotify_artist_catalog["artist_individual_norm"] = (
    spotify_artist_catalog["artist_individual"]
    .apply(normalize_person)
)


# ------------------------------------------------------------
# 6. Matching worker → artista Spotify
# ------------------------------------------------------------

worker_matches = workers_exploded.merge(
    spotify_artist_catalog,
    left_on="worker_norm",
    right_on="artist_individual_norm",
    how="inner"
)


# ------------------------------------------------------------
# 7. Resultados de cobertura
# ------------------------------------------------------------

worker_records_matched = (
    worker_matches["grammy_record_id"]
    .nunique()
)

worker_people_matched = (
    worker_matches["worker_norm"]
    .nunique()
)

worker_match_rows = len(worker_matches)


print("\n--- MATCHING WORKERS → SPOTIFY ---")

print(
    f"Registros Grammy sin artist: "
    f"{len(workers_only)}"
)

print(
    f"Registros Grammy recuperados mediante workers: "
    f"{worker_records_matched}"
)

print(
    f"Personas distintas encontradas en Spotify: "
    f"{worker_people_matched}"
)

print(
    f"Filas de matching generadas: "
    f"{worker_match_rows}"
)

if len(workers_only) > 0:
    print(
        f"Cobertura adicional sobre registros sin artist: "
        f"{worker_records_matched / len(workers_only) * 100:.2f}%"
    )


# ------------------------------------------------------------
# 8. Cobertura adicional sobre todo Grammy
# ------------------------------------------------------------

current_grammy_matches = (
    matches["grammy_record_id"]
    .nunique()
)

additional_records = (
    set(worker_matches["grammy_record_id"])
    - set(matches["grammy_record_id"])
)

print("\n--- COBERTURA ADICIONAL REAL ---")

print(
    f"Registros Grammy cubiertos actualmente por artist: "
    f"{current_grammy_matches}"
)

print(
    f"Registros nuevos encontrados exclusivamente mediante workers: "
    f"{len(additional_records)}"
)

potential_combined_coverage = (
    current_grammy_matches +
    len(additional_records)
)

print(
    f"Cobertura potencial combinada: "
    f"{potential_combined_coverage} / {len(grammy)} "
    f"({potential_combined_coverage / len(grammy) * 100:.2f}%)"
)


# ------------------------------------------------------------
# 9. ¿Qué roles están generando los matches?
# ------------------------------------------------------------

print("\n--- ROLES CON MATCH EN SPOTIFY ---")

print(
    worker_matches["role"]
    .value_counts(dropna=False)
    .head(20)
)


# ------------------------------------------------------------
# 10. Casos recuperados exclusivamente mediante workers
# ------------------------------------------------------------

exclusive_worker_matches = worker_matches[
    worker_matches["grammy_record_id"].isin(additional_records)
].copy()

print("\n--- EJEMPLOS DE MATCHES RECUPERADOS SOLO POR WORKERS ---")

print(
    exclusive_worker_matches[
        [
            "grammy_record_id",
            "year",
            "category",
            "nominee",
            "worker_raw",
            "role",
            "artist_individual"
        ]
    ]
    .drop_duplicates()
    .head(30)
    .to_string(index=False)
)

print("\n" + "=" * 70)

PRUEBA 7 — COBERTURA ADICIONAL MEDIANTE WORKERS

--- DISPONIBILIDAD DE WORKERS ---
Registros Grammy sin artist: 1840
Registros sin artist pero con workers: 1654
Registros sin artist ni workers: 186

--- PERSONAS EXTRAÍDAS DE WORKERS ---
Registros de personas/roles extraídos: 3117
Personas normalizadas distintas: 2048

Distribución de roles:
role
NaN                                304
producer                           172
conductor                          123
artists                            103
artist                              82
producers                           80
composer                            79
soloist                             74
compilation producers               46
engineers                           46
conductor)                          42
engineer                            33
ensembles                           29
soloists                            28
songwriters                         26
principal soloists                  20
composer (John Williams)    

In [20]:
# ============================================================
# PRUEBA 8 — VALIDACIÓN SEMÁNTICA DE MATCHES MEDIANTE WORKERS
# ============================================================

# Roles encontrados que potencialmente pueden representar
# una relación directa con un artista/intérprete.
candidate_roles = [
    "artist",
    "artists",
    "soloist",
    "vocalist",
    "performer",
    "musician",
    "singer",
]

# Normalizar roles
worker_matches["role_norm"] = (
    worker_matches["role"]
    .fillna("")
    .str.lower()
    .str.strip()
)

print("=" * 70)
print("PRUEBA 8 — VALIDACIÓN SEMÁNTICA DE MATCHES MEDIANTE WORKERS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Distribución completa de roles con match
# ------------------------------------------------------------

print("\n--- ROLES CON MATCH EN SPOTIFY ---")

role_summary = (
    worker_matches.groupby("role_norm")
    .agg(
        match_rows=("grammy_record_id", "size"),
        grammy_records=("grammy_record_id", "nunique"),
        people=("worker_norm", "nunique")
    )
    .sort_values("grammy_records", ascending=False)
)

print(role_summary.head(30).to_string())


# ------------------------------------------------------------
# 2. Matches asociados a roles potencialmente artísticos
# ------------------------------------------------------------

artist_role_matches = worker_matches[
    worker_matches["role_norm"].isin(candidate_roles)
].copy()

print("\n--- MATCHES CON ROLES POTENCIALMENTE ARTÍSTICOS ---")
print(
    f"Filas de matching: {len(artist_role_matches)}"
)
print(
    f"Personas distintas: "
    f"{artist_role_matches['worker_norm'].nunique()}"
)
print(
    f"Registros Grammy distintos: "
    f"{artist_role_matches['grammy_record_id'].nunique()}"
)

print(
    f"Registros Grammy recuperados exclusivamente mediante "
    f"roles potencialmente artísticos: "
    f"{artist_role_matches['grammy_record_id'].isin(additional_records).sum()}"
)


# ------------------------------------------------------------
# 3. Comparar contra roles técnicos
# ------------------------------------------------------------

technical_keywords = [
    "producer",
    "producer",
    "engineer",
    "mixer",
    "mastering",
    "conductor",
    "composer",
    "songwriter",
    "arranger",
    "remixer",
]

technical_matches = worker_matches[
    worker_matches["role_norm"].apply(
        lambda x: any(keyword in x for keyword in technical_keywords)
    )
].copy()

print("\n--- MATCHES CON ROLES TÉCNICOS / PRODUCCIÓN ---")
print(
    f"Filas de matching: {len(technical_matches)}"
)
print(
    f"Personas distintas: "
    f"{technical_matches['worker_norm'].nunique()}"
)
print(
    f"Registros Grammy distintos: "
    f"{technical_matches['grammy_record_id'].nunique()}"
)


# ------------------------------------------------------------
# 4. Cobertura adicional realmente utilizable
# ------------------------------------------------------------

additional_artist_records = set(
    artist_role_matches["grammy_record_id"]
) - set(matches["grammy_record_id"])

print("\n--- COBERTURA ADICIONAL POTENCIALMENTE UTILIZABLE ---")

print(
    f"Registros Grammy cubiertos por artist: "
    f"{current_grammy_matches}"
)

print(
    f"Registros nuevos mediante workers con rol artístico: "
    f"{len(additional_artist_records)}"
)

combined_artist_coverage = (
    current_grammy_matches + len(additional_artist_records)
)

print(
    f"Cobertura combinada: "
    f"{combined_artist_coverage} / {len(grammy)} "
    f"({combined_artist_coverage / len(grammy) * 100:.2f}%)"
)


# ------------------------------------------------------------
# 5. Ejemplos de matches adicionales
# ------------------------------------------------------------

exclusive_artist_matches = artist_role_matches[
    artist_role_matches["grammy_record_id"].isin(
        additional_artist_records
    )
].copy()

print("\n--- EJEMPLOS DE MATCHES ADICIONALES ---")

print(
    exclusive_artist_matches[
        [
            "grammy_record_id",
            "year",
            "category",
            "nominee",
            "worker_raw",
            "role",
            "artist_individual"
        ]
    ]
    .drop_duplicates()
    .head(30)
    .to_string(index=False)
)

print("\n" + "=" * 70)

PRUEBA 8 — VALIDACIÓN SEMÁNTICA DE MATCHES MEDIANTE WORKERS

--- ROLES CON MATCH EN SPOTIFY ---
                                           match_rows  grammy_records  people
role_norm                                                                    
conductor                                          49              49      24
artist                                             47              47      15
                                                   32              32      30
composer                                           28              28      19
soloist                                            28              28      19
composer (john williams)                           20              20       1
artists                                            18              15      10
principal soloists                                  9               7       9
conductor (chicago symphony orchestra)              6               6       4
producer                                      

In [21]:
print("Registros Grammy cubiertos por artist:", current_grammy_matches)
print(
    "Registros nuevos mediante workers con rol artístico:",
    len(additional_artist_records)
)
print(
    "Cobertura combinada:",
    current_grammy_matches + len(additional_artist_records),
    "/",
    len(grammy),
    f"({(current_grammy_matches + len(additional_artist_records)) / len(grammy) * 100:.2f}%)"
)

Registros Grammy cubiertos por artist: 1295
Registros nuevos mediante workers con rol artístico: 90
Cobertura combinada: 1385 / 4810 (28.79%)
